In [ ]:
from datetime import datetime
from typing import List, Dict, Optional
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import sys
print(sys.executable)          # Which Python your notebook uses

import plotly
print("plotly:", plotly.__version__)

from kaleido import write_fig_sync

print("kaleido OK")

# Default contributors (exact names as requested)
CONTRIBUTORS = [
    "M. Yusuf Aykut",
    "İbrahim Abu Shawish",
    "Samet Ertuğrul Kurum",
    "Samet Baturay",
]

# Example project tasks; customize as needed
PROJECT_TASKS: List[Dict] = [
    {
        "Task": "Mimari",
        "Group": "Mimari",
        "Contributor": "M. Yusuf Aykut",
        "Start": "2025-12-05",
        "Finish": "2025-12-25",
        "Progress": 100,
    },
    {
        "Task": "Literatür Taraması",
        "Group": "Literatür Taraması",
        "Contributor": "Samet Ertuğrul Kurum",
        "Start": "2025-12-10",
        "Finish": "2025-12-21",
        "Progress": 75,
    },
    {
        "Task": "Model Eğtimi",
        "Group": "Model Eğtimi",
        "Contributor": ["M. Yusuf Aykut", "İbrahim Abu Shawish"],
        "Start": "2025-12-15",
        "Finish": "2025-12-28",
        "Progress": 60,
    },
    {
        "Task": "Makale yazısı",
        "Group": "Makale yazısı",
        "Contributor": ["M. Yusuf Aykut", "İbrahim Abu Shawish","Samet Ertuğrul Kurum","Samet Baturay"],
        "Start": "2025-12-20",
        "Finish": "2026-01-02",
        "Progress": 40,
    },
    {
        "Task": "Teknik müdahale",
        "Group": "Teknik müdahale",
        "Contributor": ["M. Yusuf Aykut", "İbrahim Abu Shawish","Samet Ertuğrul Kurum","Samet Baturay"],
        "Start": "2025-12-25",
        "Finish": "2026-01-07",
        "Progress": 30,
    },
    {
        "Task": "Test",
        "Group": "Test",
        "Contributor": "İbrahim Abu Shawish",
        "Start": "2025-12-10",
        "Finish": "2025-12-16",
        "Progress": 20,
    },
    {
        "Task": "Documentation",
        "Group": "Documentation",
        "Contributor": "Samet Baturay",
        "Start": "2026-01-03",
        "Finish": "2026-01-14",
        "Progress": 10,
    },
]


def _parse_dates(df, *cols):
    for col in cols:
        df[col] = pd.to_datetime(
            df[col],
            format="mixed",   # lets pandas infer per element
            dayfirst=False,
            errors="raise",   # still fail loudly on truly bad values
        )
    return df


def _expand_contributors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure each row has exactly one contributor. If a task has multiple
    contributors (list), duplicate the row per contributor.
    """
    df = df.copy()
    df["Contributor"] = df["Contributor"].apply(
        lambda v: v if isinstance(v, list) else [v]
    )
    df = df.explode("Contributor").reset_index(drop=True)
    return df


def _validate_tasks(tasks: List[Dict], contributors: List[str]) -> pd.DataFrame:
    df = pd.DataFrame(tasks)
    required_cols = {"Task", "Group", "Contributor", "Start", "Finish"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing task fields: {missing}")

    df = _expand_contributors(df)
    unknown_contribs = set(df["Contributor"]) - set(contributors)
    if unknown_contribs:
        raise ValueError(f"Unknown contributors in tasks: {unknown_contribs}")

    df = _parse_dates(df, "Start", "Finish")
    if (df["Finish"] < df["Start"]).any():
        bad = df[df["Finish"] < df["Start"]]
        raise ValueError(f"Finish before Start for tasks: {bad['Task'].tolist()}")

    df["DurationDays"] = (df["Finish"] - df["Start"]).dt.days
    df["Progress"] = df.get("Progress", pd.Series([None] * len(df)))
    df["Hover"] = df.apply(
        lambda r: f"{r['Task']} | {r['Group']} | {r['Contributor']}\n"
                  f"{r['Start'].date()} → {r['Finish'].date()} ({r['DurationDays']}d)\n"
                  f"Progress: {r['Progress']}%", axis=1
    )
    return df


def build_gantt(
    tasks: List[Dict],
    contributors: Optional[List[str]] = None,
    color_by: str = "Contributor",  # "Contributor" or "Group"
    contributor_filter: Optional[List[str]] = None,
    show_progress_overlay: bool = True,
    title: str = "(Temsili!) Gantt Chart (Geçici!)",
) -> go.Figure:
    contributors = contributors or CONTRIBUTORS
    df = _validate_tasks(tasks, contributors)

    if contributor_filter:
        df = df[df["Contributor"].isin(contributor_filter)].copy()
        if df.empty:
            raise ValueError("No tasks after applying contributor_filter.")

    # Create a display task name that includes contributor for multi-contributor tasks
    df["TaskDisplay"] = df.apply(
        lambda r: f"{r['Task']} ({r['Contributor']})", axis=1
)

    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="TaskDisplay",
        color=color_by,
        hover_name="Task",
        hover_data={
            "Hover": True,
            "Start": True,
            "Finish": True,
            "DurationDays": True,
            "Progress": True,
            "Task": False,
            "TaskDisplay": False,
        },
)
    fig.update_traces(hovertemplate="%{customdata[0]}")
    df_sorted = df.sort_values(by=["Start", "Finish"])
    fig.update_yaxes(categoryorder="array", categoryarray=df_sorted["TaskDisplay"].tolist())
    fig.update_layout(
        title=title,
        xaxis_title="Timeline",
        yaxis_title="Tasks",
        bargap=0.2,
        legend_title=color_by,
        hovermode="closest",
        template="plotly_white",
)

    if show_progress_overlay and "Progress" in df.columns and df["Progress"].notnull().any():
        overlay_traces = []
        for _, r in df.iterrows():
            if pd.isna(r["Progress"]):
                continue
            progress_ratio = max(0.0, min(1.0, float(r["Progress"]) / 100.0))
            progress_end = r["Start"] + (r["Finish"] - r["Start"]) * progress_ratio
            overlay_traces.append(go.Bar(
                x=[(progress_end - r["Start"]).days],
                y=[r["TaskDisplay"]],
                base=r["Start"],
                orientation="h",
                marker=dict(color="rgba(0,0,0,0.15)"),
                showlegend=False,
                hoverinfo="skip",
))
        for t in overlay_traces:
            fig.add_trace(t)

    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7, label="1w", step="day", stepmode="backward"),
                    dict(count=14, label="2w", step="day", stepmode="backward"),
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(step="all")
                ])
),
            rangeslider=dict(visible=True),
            type="date"
)
)

    return fig


def build_contribution_sankey(
    tasks: List[Dict],
    contributors: Optional[List[str]] = None,
    title: str = "(Temsili!) Contribution Flow (Geçici!)"
) -> go.Figure:
    contributors = contributors or CONTRIBUTORS
    df = _validate_tasks(tasks, contributors)

    groups = sorted(df["Group"].unique().tolist())
    nodes = contributors + groups
    node_index = {name: i for i, name in enumerate(nodes)}

    agg = df.groupby(["Contributor", "Group"])["DurationDays"].sum().reset_index()

    links = dict(
        source=[node_index[c] for c in agg["Contributor"]],
        target=[node_index[g] for g in agg["Group"]],
        value=agg["DurationDays"].tolist(),
        label=[f"{c} → {g}: {v}d" for c, g, v in agg.values]
)

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color=["#4C78A8"] * len(contributors) + ["#F58518"] * len(groups),
) ,
        link=links
)])
    fig.update_layout(title_text=title, font_size=12, template="plotly_white")
    return fig


def _render(color_by, selected_contribs, show_progress, show_sankey):
    clear_output(wait=True)
    print("Controls (change values to update):")
    display(ui_box)
    gantt = build_gantt(
        PROJECT_TASKS,
        contributors=CONTRIBUTORS,
        color_by=color_by,
        contributor_filter=selected_contribs if selected_contribs else None,
        show_progress_overlay=show_progress,
        title="Project Gantt Chart"
)
    gantt.show()
    write_fig_sync(gantt, path="gantt.png", opts={"format": "png"})
    if show_sankey:
        sankey = build_contribution_sankey(PROJECT_TASKS, contributors=CONTRIBUTORS)
        sankey.show() # requires `pip install -q kaleido`
        write_fig_sync(sankey, path="sankey.png", opts={"format": "png"})

color_dropdown = widgets.Dropdown(
    options=["Contributor", "Group"],
    value="Contributor",
    description="Color by:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="200px"),
)

contrib_select = widgets.SelectMultiple(
    options=CONTRIBUTORS,
    value=(),
    description="Contributors:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="250px", height="130px"),
)

progress_toggle = widgets.Checkbox(
    value=True,
    description="Show progress overlay",
    indent=False,
)

sankey_toggle = widgets.Checkbox(
    value=True,
    description="Show Sankey flow",
    indent=False,
)

ui_box = widgets.VBox([
    widgets.HBox([color_dropdown, contrib_select]),
    widgets.HBox([progress_toggle, sankey_toggle]),
])

def main():
    out = widgets.interactive_output(
        _render,
        {
            "color_by": color_dropdown,
            "selected_contribs": contrib_select,
            "show_progress": progress_toggle,
            "show_sankey": sankey_toggle,
        }
)
    display(ui_box, out)

# Run the interactive UI in the notebook
main()

c:\Users\Pc\AppData\Local\Programs\Python\Python311\python.exe
plotly: 6.5.0
kaleido OK


Output()